In [0]:
# Set Up and load 
from pyspark.sql.functions import col, to_date, when, lit

# Authenticate
storage_account_name = "{storage_account_name}"
storage_account_key = "{storage_account_key}"
spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)

# Define paths
base_path = f"abfss://dubairealestatedata@{storage_account_name}.dfs.core.windows.net/bronze"
silver_path = f"abfss://dubairealestatedata@{storage_account_name}.dfs.core.windows.net/silver"

# Reading bronze CSVs
print("Reading Bronze files")
df_transactions_bronze = spark.read.format("csv").option("header", "true").load(f"{base_path}/transactions/landing/Transactions.csv")
df_projects_bronze = spark.read.format("csv").option("header", "true").load(f"{base_path}/projects/landing/Projects.csv")
df_units_bronze = spark.read.format("csv").option("header", "true").load(f"{base_path}/units/landing/Units.csv")

print("Bronze data loaded and silver container configured")

In [0]:
# Understanding the data
df_transactions_bronze.display()
df_projects_bronze.display()
df_units_bronze.display()

In [0]:
# Clean up the Projects (Dimensions)
df_projects_clean = df_projects_bronze.select(col("project_number").cast("int"),col("project_status"),col("percent_completed").cast("double"),
                                      # FIX the date format to 'dd-MM-yyyy'
                                      to_date(col("project_start_date"),"dd-MM-yyyy").alias("start_date"),
                                      to_date(col("completion_date"),"dd-MM-yyyy").alias("completion_date"),
                                      col("master_project_en").alias("master_community")).dropDuplicates(["project_number"])
df_projects_clean.display()                               

In [0]:
# Clean the Transactions (Fact Table)
# -------------------------------------------------------------------------
df_transactions_clean = df_transactions_bronze.select(
col("transaction_id"),

# FIX: Date format set to 'dd-MM-yyyy'
to_date(col("instance_date"), "dd-MM-yyyy").alias("transaction_date"),

col("procedure_name_en").alias("transactions_type"),
col("trans_group_en").alias("transactions_group"),

# Location & Project Info
col("area_name_en").alias("area_name"),
col("building_name_en").alias("building_name"),
col("project_number").cast("int"),

# --- We get the English Name from here ---
col("project_name_en").alias("project_name"),
col("nearest_landmark_en").alias("landmark"),

# Property Details
col("property_type_en").alias("property_type"),
col("property_sub_type_en").alias("property_subtype"),
col("rooms_en").alias("room_type"), # e.g., "1 B/R"

# Metrics
col("procedure_area").cast("double").alias("area_sqft"),
col("actual_worth").cast("double").alias("price"),
col("meter_sale_price").cast("double")
)

# Filter garbage data
df_transactions_clean = df_transactions_clean.filter(col("price") > 0)

print("✅ Transactions Cleaned.")
df_transactions_clean.display()

In [0]:
# Join & Enrich (Silver Master Table)
# -------------------------------------------------------------------------

# 1. Join Transactions with Projects
df_enriched = df_transactions_clean.join(
df_projects_clean,
on="project_number",
how="left"
)

# 2. Enrich: Calculate Price Per SqFt
df_enriched = df_enriched.withColumn(
"price_per_sqft",
when(col("area_sqft") > 0, col("price") / col("area_sqft")).otherwise(0)
)

# 3. Enrich: Is Off-Plan?
df_enriched = df_enriched.withColumn(
"is_offplan_flag",
when(
(col("transaction_date") < col("completion_date")) | (col("percent_completed") < 100),
lit(True)
).otherwise(lit(False))
)

print("✅ Data Enriched & Joined.")
display(df_enriched.select("transaction_date", "project_name", "room_type", "price", "project_status","is_offplan_flag").limit(10))

In [0]:
#  Write to Silver Layer (Physical Files)
# -------------------------------------------------------------------------

# 1. Write the Master Transactions Table
print("Writing Transactions to Silver...")
df_enriched.write.format("delta").mode("overwrite").save(f"{silver_path}/transactions")

# 2. Write the Clean Units Table
print("Writing Units to Silver...")
df_units_clean = df_units_bronze.select(
col("property_id"), # Unique Key
col("project_id").cast("int").alias("project_number"), # Rename to match other tables
col("unit_number"),

# Metrics
col("rooms").alias("bedrooms"), # Numeric (e.g., "1")
col("rooms_en").alias("room_type"),# Text (e.g., "1 B/R")

# --- FIX: Use 'actual_area' ---
col("actual_area").cast("double").alias("unit_area")

).dropDuplicates(["property_id"]) # Deduplicate

df_units_clean.write.format("delta").mode("overwrite").save(f"{silver_path}/units")

print("🎉 SUCCESS: Silver Files Written to Azure.")

In [0]:
# CELL 6: Verification (Check Data Exists)
# -------------------------------------------------------------------------
print("🔍 Verifying Silver Layer...")

try:
# 1. Load the new Delta Tables directly from the file path
    df_silver_transactions = spark.read.format("delta").load(f"{silver_path}/transactions")
    df_silver_units = spark.read.format("delta").load(f"{silver_path}/units")

# 2. Print Counts
    count_transactions = df_silver_transactions.count()
    count_units = df_silver_units.count()

    print(f"✅ VERIFIED: Transactions Table contains {count_transactions} rows.")
    print(f"✅ VERIFIED: Units Table contains {count_units} rows.")

# 3. Display the data (This is your visual check)
    if count_transactions > 0:
        print("🚀 Success! Here is a preview of your clean Silver data:")
        display(df_silver_transactions.limit(5))
    else:
        print("⚠️ WARNING: Tables exist but are empty.")

except Exception as e:
        print(f"❌ ERROR: Could not read Silver tables. {e}")